# Модуль 8. Регулярные выражения
## Курс «Программирование на Python с нуля»


Занятия 22–23 (8 академических часов: 4 ч теории + 4 ч практики)

**Среда:** Google Colab  
**Язык:** русский  
**Пререквизиты:** модули 1–7 (переменные, условия, циклы, строки, списки, словари, функции, модули, файлы)

> **Структура блокнота:**
> - Занятие 22: основы регулярных выражений — метасимволы, символьные классы, квантификаторы, функции `re.search`, `re.findall`, `re.sub`, `re.split`.
> - Занятие 23: применение регулярных выражений — группы, якоря, флаги, компиляция шаблонов, жадные и нежадные квантификаторы.
>
> Каждая практика снабжена автопроверкой. Домашние задания и задания «со звёздочкой» — в конце.

## Цели модуля

После этого модуля вы сможете:

- объяснять, что такое регулярное выражение и зачем оно нужно при обработке текста;
- использовать модуль `re` и функции `search`, `match`, `findall`, `sub`, `split`;
- применять метасимволы `\d`, `\w`, `\s`, `.` и их отрицания `\D`, `\W`, `\S`;
- задавать количество повторений квантификаторами `*`, `+`, `?`, `{m,n}`;
- строить наборы символов `[...]` и диапазоны `[a-z]`, `[0-9]`;
- использовать якоря `^`, `$`, `\b` и альтернацию `|`;
- выделять и извлекать группы `(...)`, в том числе именованные `(?P<name>...)`;
- управлять поведением шаблонов флагами `re.IGNORECASE`, `re.MULTILINE`, `re.DOTALL`;
- компилировать шаблоны через `re.compile` для повторного использования;
- решать лингвистические задачи: поиск email, хештегов, чисел, очистка текста, разбиение на предложения;
- избегать типичных ошибок: несырые строки, нежадные квантификаторы, забытая проверка `None`.

## 🛠 Утилита для автопроверок

Следующая ячейка определяет вспомогательные функции. **Запустите её один раз перед началом практики.**

In [ ]:
# Утилиты для автоматической проверки заданий.
# Запустите эту ячейку один раз.


def check_value(name, actual, expected):
    """Проверяет, что переменная равна ожидаемому значению."""
    if actual == expected:
        print(f"✅ Задание пройдено: {name} = {actual!r}")
        return True
    else:
        print(f"❌ Ошибка: ожидалось {name} = {expected!r}, получено {actual!r}")
        return False


def check_function(func, cases):
    """
    Проверяет функцию на наборе тестовых случаев.
    cases — список кортежей (args, expected).
    """
    for case in cases:
        try:
            if len(case) == 2:
                args, expected = case
                actual = func(*args) if isinstance(args, tuple) else func(args)
            else:
                args, kwargs, expected = case
                actual = func(*args, **kwargs)
        except Exception as e:
            print(f"❌ Ошибка при вызове {func.__name__}: {type(e).__name__}: {e}")
            return False
        if actual != expected:
            print(f"❌ Неверный результат: {func.__name__}{args} → {actual!r}, ожидалось {expected!r}")
            return False
    print(f"✅ Функция {func.__name__} прошла все тесты")
    return True


print("Утилиты автопроверки загружены.")

---
# Часть 1. Основы регулярных выражений

## 1.1. Зачем нужны регулярные выражения

**Регулярное выражение** (regular expression, regex, «регэксп») — это строка-шаблон, которая описывает **множество** строк. С её помощью одной записью можно найти все адреса электронной почты, все номера телефонов, все слова с определённым суффиксом и т. д.

Обычные строковые методы решают лишь простые задачи:

    "hello".startswith("h")     # только проверка начала строки
    "a-b-c".split("-")          # только разделение по одному символу

А regex умеет искать **по образцу**: не «ровно эта строка», а «любая строка вида …».

### 1.1.1. Пример задачи без regex и с regex

**Задача:** извлечь все email-адреса из текста.

Без regex пришлось бы писать много кода с проверками на `@` и `.`. С regex это одна строка:

    import re
    emails = re.findall(r"[\w.+-]+@[\w-]+\.[\w.-]+", text)

### 1.1.2. Модуль `re`

Модуль `re` — это стандартная библиотека Python для работы с регулярными выражениями. Его не нужно устанавливать, достаточно импортировать:

    import re

## 1.2. Основные функции модуля `re`

| Функция | Что делает | Возвращает |
|---------|------------|------------|
| `re.search(pattern, text)` | Ищет первое вхождение шаблона в любом месте текста | Match-объект или `None` |
| `re.match(pattern, text)` | Ищет шаблон только в начале текста | Match-объект или `None` |
| `re.fullmatch(pattern, text)` | Проверяет, что вся строка совпадает с шаблоном | Match-объект или `None` |
| `re.findall(pattern, text)` | Ищет все вхождения шаблона | Список строк (или кортежей, если есть группы) |
| `re.finditer(pattern, text)` | Ищет все вхождения, возвращает итератор Match-объектов | Итератор |
| `re.sub(pattern, repl, text)` | Заменяет все вхождения шаблона на `repl` | Новая строка |
| `re.split(pattern, text)` | Разделяет текст по вхождениям шаблона | Список строк |

### 1.2.1. `search` возвращает Match-объект

`re.search` возвращает объект `Match`, у которого есть полезные методы:

- `match.group()` — найденная строка;
- `match.start()` — индекс начала вхождения;
- `match.end()` — индекс конца вхождения;
- `match.span()` — кортеж `(start, end)`.

Если вхождение не найдено — возвращается `None`, поэтому результат нужно проверять.

In [ ]:
# Пример 1. re.search и Match-объект
import re

text = "Email: anna@example.com, phone: 12345"
match = re.search(r"\d+", text)

print("match:", match)
print("group():", match.group())
print("start():", match.start())
print("end():", match.end())
print("span():", match.span())

# Если ничего не найдено — вернётся None
print("нет вхождения:", re.search(r"xyz", text))

## 1.3. Символьные классы

Символьные классы описывают **один символ** определённого типа.

| Символ | Что означает |
|--------|--------------|
| `\d` | любая цифра (0–9) |
| `\w` | любая «словесная» буква: латиница, цифра, подчёркивание |
| `\s` | любой пробельный символ: пробел, табуляция, перевод строки |
| `.` | любой символ, кроме перевода строки |
| `\D` | НЕ цифра |
| `\W` | НЕ словесный символ |
| `\S` | НЕ пробельный символ |

**Важно:** в шаблоне `\d`, `\w`, `\s` используется обратный слэш. Чтобы Python не интерпретировал его как escape-последовательность, используйте **сырые строки** `r"..."`:

    r"\d"    # правильно: шаблон — «цифра»

**Правило:** всегда пишите шаблоны regex как сырые строки `r"..."`.

In [ ]:
# Пример 2. Символьные классы
import re

print(re.findall(r"\d", "I have 2 cats and 3 dogs"))      # ['2', '3']
print(re.findall(r"\d+", "I have 2 cats and 3 dogs"))     # ['2', '3']
print(re.findall(r"\w+", "Hello, world! 2024"))           # ['Hello', 'world', '2024']
print(re.findall(r"\s+", "a b  c   d"))                   # [' ', '  ', '   ']
print(re.findall(r"\D+", "a1b2c3"))                       # ['a', 'b', 'c']

## 1.4. Квантификаторы

Квантификатор указывает, **сколько раз** должен повториться предыдущий символ или группа.

| Квантификатор | Значение | Пример | Что находит |
|---------------|----------|--------|-------------|
| `*` | 0 или более раз | `ab*` | `a`, `ab`, `abb` |
| `+` | 1 или более раз | `ab+` | `ab`, `abb` |
| `?` | 0 или 1 раз | `ab?` | `a`, `ab` |
| `{m}` | ровно m раз | `\d{3}` | три цифры |
| `{m,n}` | от m до n раз | `\d{2,4}` | две–четыре цифры |
| `{m,}` | m и более раз | `\d{2,}` | две и более цифр |

In [ ]:
# Пример 3. Квантификаторы
import re

print(re.findall(r"colou?r", "color colour"))            # ['color', 'colour']
print(re.findall(r"\d{2,4}", "1 22 333 4444 55555"))     # ['22', '333', '4444', '5555']
print(re.findall(r"ba+", "b ba baa baaa"))               # ['ba', 'baa', 'baaa']
print(re.findall(r"ba*", "b ba baa baaa"))               # ['b', 'ba', 'baa', 'baaa']

## 1.5. Наборы символов `[...]`

Квадратные скобки задают **набор** допустимых символов для одной позиции.

| Шаблон | Что находит |
|--------|-------------|
| `[abc]` | любой из символов `a`, `b`, `c` |
| `[a-z]` | любая строчная латинская буква |
| `[A-Z]` | любая заглавная латинская буква |
| `[0-9]` | любая цифра (то же, что `\d`) |
| `[a-zA-Z]` | любая латинская буква (строчная или заглавная) |
| `[^abc]` | любой символ, КРОМЕ `a`, `b`, `c` |
| `[aeiou]` | любая гласная |

**Важно:** внутри `[]` большинство метасимволов теряют специальное значение, а `^` в начале означает отрицание.

In [ ]:
# Пример 4. Наборы символов
import re

print(re.findall(r"[aeiou]", "hello world"))          # ['e', 'o', 'o']
print(re.findall(r"[0-9]", "page 42, line 7"))        # ['4', '2', '7']
print(re.findall(r"[^0-9]", "a1b2"))                  # ['a', 'b']
print(re.findall(r"[A-Z][a-z]+", "Anna and Ivan"))    # ['Anna', 'Ivan']

## Практика 1. Извлечение чисел

**Описание:** дана строка `text`. С помощью `re.findall` найдите все **числа** (последовательности цифр). Сохраните результат в переменную `numbers`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: numbers = ['2024', '42', '7']
```

In [ ]:
# Практика 1. Извлечение чисел
import re

text = "В 2024 году было 42 студента и 7 групп"
numbers = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("numbers", numbers, ["2024", "42", "7"])

<details>
<summary>Подсказка</summary>

Используйте `re.findall(r"\d+", text)`.

</details>

## Практика 2. Извлечение email-адресов

**Описание:** дана строка `text`. Извлеките все email-адреса. Сохраните результат в переменную `emails`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: emails = ['anna@example.com', 'ivan.teacher@mgpu.edu']
```

In [ ]:
# Практика 2. Извлечение email-адресов
import re

text = "Напишите на anna@example.com или ivan.teacher@mgpu.edu для записи"
emails = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("emails", emails, ["anna@example.com", "ivan.teacher@mgpu.edu"])

<details>
<summary>Подсказка</summary>

Шаблон email: `r"[\w.+-]+@[\w-]+\.[\w.-]+"`.

</details>

## Практика 3. Извлечение хештегов

**Описание:** дана строка `text`. Извлеките все хештеги (слова, начинающиеся с `#`). Сохраните результат в переменную `hashtags`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: hashtags = ['#python', '#NLP', '#english', '#ai_tools']
```

In [ ]:
# Практика 3. Извлечение хештегов
import re

text = "#python #NLP and #english learning with #ai_tools"
hashtags = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("hashtags", hashtags, ["#python", "#NLP", "#english", "#ai_tools"])

## Практика 4. Удаление цифр

**Описание:** дана строка `text`. Удалите из неё **все цифры** с помощью `re.sub`. Сохраните результат в переменную `no_digits`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: no_digits = 'abcd'
```

In [ ]:
# Практика 4. Удаление цифр
import re

text = "a1b2c3d4"
no_digits = ""

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("no_digits", no_digits, "abcd")

<details>
<summary>Подсказка</summary>

Используйте `re.sub(r"\d", "", text)`.

</details>

## Практика 5. Разбиение строки по пробелам

**Описание:** дана строка `text` с разным количеством пробелов между словами. Разделите её на слова с помощью `re.split`. Сохраните результат в переменную `words`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: words = ['one', 'two', 'three']
```

In [ ]:
# Практика 5. Разбиение строки по пробелам
import re

text = "one  two   three"
words = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("words", words, ["one", "two", "three"])

<details>
<summary>Подсказка</summary>

Используйте `re.split(r"\s+", text)`.

</details>

---
# Часть 2. Применение регулярных выражений

## 2.1. Якоря: `^`, `$`, `\b`

Якоря не «съедают» символы, а указывают **позицию** в строке.

| Якорь | Позиция |
|-------|---------|
| `^` | начало строки |
| `$` | конец строки |
| `\b` | граница слова (переход между `\w` и не-`\w`) |
| `\B` | НЕ граница слова |

In [ ]:
# Пример 5. Якоря
import re

print(re.findall(r"^Hello", "Hello world"))     # ['Hello']
print(re.findall(r"^world", "Hello world"))     # []
print(re.findall(r"world$", "Hello world"))     # ['world']

# \b выделяет целые слова
print(re.findall(r"\bcat\b", "cat catalog bobcat cat"))   # ['cat', 'cat']

## 2.2. Альтернация `|`

Вертикальная черта `|` означает «или»:

    re.findall(r"cat|dog", "I have a cat and a dog")   # ['cat', 'dog']

Альтернацию удобно комбинировать с группами и границами слов.

## 2.3. Группы `(...)`

Круглые скобки объединяют часть шаблона в **группу**. Группы нужны, чтобы:

- применить квантификатор к целой части: `(ab)+` — «ab» повторяется;
- извлечь отдельную часть совпадения;
- использовать обратные ссылки.

### 2.3.1. Извлечение частей совпадения

Если в шаблоне есть группы, `re.search(...).group(n)` возвращает конкретную группу, а `re.findall` — кортежи значений групп:

    m = re.search(r"(\d{2})-(\d{2})-(\d{4})", "Дата: 15-11-2026")
    m.group(1)   # '15'
    m.group(2)   # '11'
    m.group(3)   # '2026'

### 2.3.2. Именованные группы `(?P<name>...)`

Группе можно дать имя и обращаться по имени:

    m = re.search(r"(?P<day>\d{2})-(?P<month>\d{2})-(?P<year>\d{4})", "15-11-2026")
    m.group("year")   # '2026'

In [ ]:
# Пример 6. Группы
import re

text = "Дата: 15-11-2026"
m = re.search(r"(\d{2})-(\d{2})-(\d{4})", text)
print("вся строка:", m.group(0))
print("день:", m.group(1))
print("месяц:", m.group(2))
print("год:", m.group(3))

# Именованные группы
m2 = re.search(r"(?P<day>\d{2})-(?P<month>\d{2})-(?P<year>\d{4})", text)
print("год (по имени):", m2.group("year"))

# findall с группами возвращает кортежи
print(re.findall(r"(\d{2})-(\d{2})-(\d{4})", text))

## 2.4. Жадные и нежадные квантификаторы

По умолчанию квантификаторы **жадные**: они захватывают как можно больше символов.

    re.findall(r"<.+>", "<b>bold</b> and <i>italic</i>")
    # ['<b>bold</b> and <i>italic</i>']   — один «слишком длинный» результат

Чтобы сделать квантификатор **нежадным** (ленивым), добавьте `?`:

    re.findall(r"<.+?>", "<b>bold</b> and <i>italic</i>")
    # ['<b>', '</b>', '<i>', '</i>']

| Жадный | Нежадный |
|--------|----------|
| `*` | `*?` |
| `+` | `+?` |
| `?` | `??` |
| `{m,n}` | `{m,n}?` |

In [ ]:
# Пример 7. Жадные и нежадные квантификаторы
import re

html = "<b>bold</b> and <i>italic</i>"
print("жадный:  ", re.findall(r"<.+>", html))
print("нежадный:", re.findall(r"<.+?>", html))

## 2.5. Флаги

Флаги меняют поведение шаблона. Передаются третьим аргументом функции или в `re.compile`.

| Флаг | Что делает |
|------|------------|
| `re.IGNORECASE` (или `re.I`) | Игнорирует регистр букв |
| `re.MULTILINE` (или `re.M`) | `^` и `$` работают для каждой строки |
| `re.DOTALL` (или `re.S`) | `.` находит и символы перевода строки |
| `re.VERBOSE` (или `re.X`) | Позволяет писать шаблон с комментариями и пробелами |

Флаги можно комбинировать через `|`: `re.IGNORECASE | re.MULTILINE`.

In [ ]:
# Пример 8. Флаги
import re

print(re.findall(r"python", "Python PYTHON python"))           # ['python']
print(re.findall(r"python", "Python PYTHON python", re.I))     # ['Python', 'PYTHON', 'python']

# MULTILINE: ^ означает начало каждой строки
text = "cat\ncar\nbat"
print(re.findall(r"^c\w+", text))         # ['cat']
print(re.findall(r"^c\w+", text, re.M))   # ['cat', 'car']

## 2.6. Компиляция шаблонов: `re.compile`

Если один и тот же шаблон используется много раз, его можно скомпилировать один раз и переиспользовать:

    pattern = re.compile(r"\d+", re.IGNORECASE)
    pattern.findall("a1 b22 c333")   # ['1', '22', '333']

У скомпилированного шаблона есть те же методы: `search`, `findall`, `sub`, `split` и т. д.

**Когда компилировать:** в циклах и при многократном использовании одного шаблона — это читаемее и немного быстрее.

In [ ]:
# Пример 9. re.compile
import re

email_pattern = re.compile(r"[\w.+-]+@[\w-]+\.[\w.-]+")

texts = [
    "Пишите: anna@example.com",
    "Контакт: ivan@mgpu.edu",
    "Нет почты",
]

for t in texts:
    print(t, "→", email_pattern.findall(t))

## Практика 6. Разбор даты

**Описание:** дана строка `date_str = "15-11-2026"`. С помощью групп извлеките **год**. Сохраните результат в переменную `year`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: year = '2026'
```

In [ ]:
# Практика 6. Разбор даты
import re

date_str = "15-11-2026"
year = ""

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("year", year, "2026")

<details>
<summary>Подсказка</summary>

Используйте `re.search(r"(\d{2})-(\d{2})-(\d{4})", date_str).group(3)`.

</details>

## Практика 7. Поиск повторяющихся слов

**Описание:** дана строка `text`. Найдите **повторяющиеся подряд** слова (например, «the the»). Сохраните список самих повторяющихся слов в переменную `dupes`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: dupes = ['the', 'on']
```

In [ ]:
# Практика 7. Повторяющиеся слова
import re

text = "the the cat sat on on the mat"
dupes = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("dupes", dupes, ["the", "on"])

<details>
<summary>Подсказка</summary>

Шаблон с обратной ссылкой: `r"\b(\w+)\s+\1\b"`.

</details>

## Практика 8. Очистка текста от пунктуации

**Описание:** дана строка `text`. Удалите из неё все знаки пунктуации (всё, что не буква/цифра/пробел). Сохраните результат в переменную `clean`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: clean = 'Hello world Python is great'
```

In [ ]:
# Практика 8. Очистка от пунктуации
import re

text = "Hello, world! Python is great."
clean = ""

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("clean", clean, "Hello world Python is great")

<details>
<summary>Подсказка</summary>

Используйте `re.sub(r"[^\w\s]", "", text)` — всё, что не буква/цифра и не пробел, заменяем на пустую строку.

</details>

## Практика 9. Перестановка частей даты

**Описание:** дана строка `date_str = "2026-11-15"` в формате `YYYY-MM-DD`. Преобразуйте её в формат `DD.MM.YYYY` с помощью `re.sub` и обратных ссылок. Сохраните результат в переменную `reordered`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: reordered = '15.11.2026'
```

In [ ]:
# Практика 9. Перестановка частей даты
import re

date_str = "2026-11-15"
reordered = ""

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("reordered", reordered, "15.11.2026")

<details>
<summary>Подсказка</summary>

`re.sub(r"(\d{4})-(\d{2})-(\d{2})", r"\3.\2.\1", date_str)`. Строку замены пишите как сырую `r"..."`.

</details>

## Практика 10. Разбиение текста на предложения

**Описание:** дана строка `text`. Разбейте её на предложения по знакам `.`, `!`, `?` и удалите пустые фрагменты. Сохраните результат в переменную `sentences`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: sentences = ['First sentence', 'Second one', 'Third']
```

In [ ]:
# Практика 10. Разбиение текста на предложения
import re

text = "First sentence. Second one! Third?"
sentences = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("sentences", sentences, ["First sentence", "Second one", "Third"])

<details>
<summary>Подсказка</summary>

`sentences = [s for s in re.split(r"[.!?]\s*", text) if s]`.

</details>

---
# Часть 3. Типичные ошибки

| Ошибка | Что происходит | Как исправить |
|--------|----------------|----------------|
| Шаблон не в сырой строке | `"\d"` может дать неожиданный результат | Пишите `r"\d"` |
| `re.search` без проверки на `None` | `AttributeError` при обращении к `.group()` | Проверяйте `if m:` перед использованием |
| Жадный квантификатор | Захватывает больше, чем нужно | Добавьте `?`: `r"<.+?>"` |
| `findall` с группами возвращает кортежи | Неожиданный формат результата | Помните: есть группа → вернутся кортежи |
| `.` не находит перевод строки | Пропуск «многострочных» совпадений | Используйте `re.DOTALL` |
| Спецсимволы в тексте | `.`, `+`, `*` имеют смысл в шаблоне | Экранируйте: `re.escape("1+1=2")` |
| `re.match` вместо `search` | Ищет только в начале строки | Используйте `search` для поиска в любом месте |
| Забыт `r` перед строкой замены | `"\1"` и `r"\1"` имеют разный смысл | Пишите строку замены как `r"..."` |

---
# Часть 4. Домашние задания (базовые)

## Задание 1. Слова длиннее 5 букв

**Описание:** дан текст `text`. Найдите все английские слова **длиной больше 5 букв** (только буквы). Используйте границы слова. Сохраните результат в переменную `long_words`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: long_words = ['elephant', 'butterfly']
```

In [ ]:
# Задание 1. Слова длиннее 5 букв
import re

text = "elephant and cat and butterfly and zebra"
long_words = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("long_words", long_words, ["elephant", "butterfly"])

<details>
<summary>Подсказка</summary>

`re.findall(r"\b[a-zA-Z]{6,}\b", text)`.

</details>

## Задание 2. Проверка орфографии по словарю

**Описание:** дан словарь `dictionary` (множество правильных слов) и текст `text`. Найдите слова, которых **нет** в словаре (слова привести к нижнему регистру). Сохраните результат в переменную `misspelled`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: misspelled = ['with', 'a', 'dog']
```

In [ ]:
# Задание 2. Проверка орфографии по словарю
import re

dictionary = {"the", "cat", "sat", "on", "mat"}
text = "The cat sat on the mat with a dog"
misspelled = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("misspelled", misspelled, ["with", "a", "dog"])

<details>
<summary>Подсказка</summary>

Извлеките слова `re.findall(r"\w+", text.lower())` и отфильтруйте те, которых нет в `dictionary`.

</details>

## Задание 3. Подсчёт email-адресов

**Описание:** дан текст `text`. Посчитайте, сколько email-адресов в нём. Сохраните результат в переменную `count`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: count = 3
```

In [ ]:
# Задание 3. Подсчёт email-адресов
import re

text = "Пишите: anna@example.com, ivan@mgpu.edu и olga@mail.ru"
count = 0

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("count", count, 3)

## Задание 4. Нормализация пробелов

**Описание:** дан текст `text` с лишними пробелами и переносами строк. Замените все последовательности пробельных символов на один пробел. Сохраните результат в переменную `normalized`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: normalized = 'Hello world Python is great'
```

In [ ]:
# Задание 4. Нормализация пробелов
import re

text = "Hello   world\nPython  is\tgreat"
normalized = ""

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("normalized", normalized, "Hello world Python is great")

<details>
<summary>Подсказка</summary>

`re.sub(r"\s+", " ", text)`.

</details>

## Задание 5. Извлечение слов в кавычках

**Описание:** дан текст `text`. Извлеките все слова, заключённые в двойные кавычки. Сохраните результат в переменную `quoted`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: quoted = ['hello', 'world']
```

In [ ]:
# Задание 5. Слова в кавычках
import re

text = 'He said "hello" and "world"'
quoted = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("quoted", quoted, ["hello", "world"])

<details>
<summary>Подсказка</summary>

`re.findall(r'"(\w+)"', text)`.

</details>

---
# Часть 5. Задания «со звёздочкой»

## Задание *1. Позиции слов

**Описание:** дан текст `text`. С помощью `re.finditer` найдите **позиции** (индексы начала и конца) всех английских слов. Сохраните список кортежей `(start, end)` в переменную `positions`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: positions = [(0, 2), (3, 8), (9, 14)]
```

In [ ]:
# Задание *1. Позиции слов
import re

text = "AI helps learn"
positions = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("positions", positions, [(0, 2), (3, 8), (9, 14)])

<details>
<summary>Подсказка</summary>

`[(m.start(), m.end()) for m in re.finditer(r"\b\w+\b", text)]`.

</details>

## Задание *2. Проверка формата даты

**Описание:** напишите функцию `is_date(s)`, которая возвращает `True`, если строка имеет формат `DD-MM-YYYY` (день и месяц — ровно две цифры, год — ровно четыре), и `False` иначе.

**Ожидаемый результат:**
```
✅ Функция is_date прошла все тесты
```

In [ ]:
# Задание *2. is_date(s)
import re

def is_date(s):
    # YOUR CODE HERE
    pass


# --- Автопроверка ---
check_function(is_date, [
    ("15-11-2026", True),
    ("2026-11-15", False),
    ("5-11-2026", False),
    ("15-11-26", False),
])

<details>
<summary>Подсказка</summary>

Используйте `re.fullmatch(r"\d{2}-\d{2}-\d{4}", s)` и проверьте результат на `None`.

</details>

## Задание *3. Извлечение URL

**Описание:** дан текст `text`. Извлеките все URL вида `http://...` и `https://...`. Сохраните результат в переменную `urls`.

**Ожидаемый вывод автопроверки:**
```
✅ Задание пройдено: urls = ['https://example.com', 'http://test.org/page']
```

In [ ]:
# Задание *3. Извлечение URL
import re

text = "Visit https://example.com or http://test.org/page and not ftp://x.com"
urls = []

# Введите ваш код ниже:
# YOUR CODE HERE


# --- Автопроверка ---
check_value("urls", urls, ["https://example.com", "http://test.org/page"])

<details>
<summary>Подсказка</summary>

`re.findall(r"https?://[\w./-]+", text)`.

</details>

## Задание *4. CamelCase → snake_case

**Описание:** напишите функцию `camel_to_snake(s)`, которая преобразует строку в стиле `camelCase` в `snake_case`. Используйте `re.sub`.

**Ожидаемый результат:**
```
✅ Функция camel_to_snake прошла все тесты
```

In [ ]:
# Задание *4. camel_to_snake(s)
import re

def camel_to_snake(s):
    # YOUR CODE HERE
    pass


# --- Автопроверка ---
check_function(camel_to_snake, [
    ("camelCaseWord", "camel_case_word"),
    ("helloWorld", "hello_world"),
    ("already_snake", "already_snake"),
])

<details>
<summary>Подсказка</summary>

`re.sub(r"(?<!^)(?=[A-Z])", "_", s).lower()`.

</details>

---
# Итоги модуля

Мы разобрали:

- **что такое регулярные выражения** и зачем они нужны при обработке текста;
- **функции модуля `re`**: `search`, `match`, `fullmatch`, `findall`, `finditer`, `sub`, `split`;
- **символьные классы** `\d`, `\w`, `\s`, `.` и их отрицания;
- **квантификаторы** `*`, `+`, `?`, `{m,n}`;
- **наборы символов** `[...]` и диапазоны;
- **якоря** `^`, `$`, `\b` и альтернацию `|`;
- **группы** `(...)` и именованные группы `(?P<name>...)`;
- **жадные и нежадные** квантификаторы;
- **флаги** `re.IGNORECASE`, `re.MULTILINE`, `re.DOTALL`;
- **компиляцию** шаблонов через `re.compile`;
- типичные ошибки и способы их избежать.

**Что дальше:** в модуле 9 перейдём к основам ООП — классам, объектам, наследованию и магическим методам.

---
# Литература

### Основная

1. **Документация Python (рус.).** Модуль `re`: https://docs.python.org/ru/3/library/re.html
2. **Документация Python (рус.).** HOWTO по регулярным выражениям: https://docs.python.org/ru/3/howto/regex.html
3. **Свейгарт А.** Автоматизация рутинных задач с помощью Python. 2-е изд. — Гл. 7 «Поиск по шаблону с помощью регулярных выражений».
4. **Лутц М.** Изучаем Python. 5-е изд. — раздел о строках и модуле `re`.

### Дополнительная

1. **Real Python.** Regular Expressions: Regexes in Python: https://realpython.com/regex-python/
2. **W3Schools.** Python RegEx: https://www.w3schools.com/python/python_regex.asp
3. **Stepik.** Курс «Поколение Python» — модуль «Регулярные выражения».
4. **regex101.com** — онлайн-песочница для отладки регулярных выражений: https://regex101.com/

### Для практики

- **Codewars** (уровень 6 kyu) — задачи на регулярные выражения.
- **HackerRank** — раздел Python → Regex.